# pass@k Run Analysis

Loads `samples.csv` and `pass_at_k.csv` from the latest run under `results/runs`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('results/runs'), Path('../results/runs')]
RUN_DIR = next((p.resolve() for p in candidates if p.exists()), candidates[0].resolve())
runs = sorted([p for p in RUN_DIR.glob('*') if p.is_dir()])
run = runs[-1] if runs else None
assert run is not None, f'No runs found under {RUN_DIR}'
run

In [ ]:
samples = pd.read_csv(run / 'samples.csv')
pass_at_k = pd.read_csv(run / 'pass_at_k.csv')
metric_cols = [c for c in pass_at_k.columns if c.startswith('pass@')]

aggregate = pass_at_k[pass_at_k['question_id'] == '__aggregate__'].set_index('model')
aggregate[metric_cols + ['num_samples', 'num_correct']].sort_values(metric_cols[0], ascending=False)

In [ ]:
ax = aggregate[metric_cols].sort_values(metric_cols[0]).plot(kind='barh', figsize=(8, 4))
ax.set_xlim(0, 1)
ax.set_xlabel('score')
ax.set_title('Model pass@k / accuracy')
plt.tight_layout()

In [ ]:
by_question = pass_at_k[pass_at_k['question_id'] != '__aggregate__'].copy()
by_question.head()

In [ ]:
hardest = by_question.groupby('question_id')[metric_cols].mean().sort_values(metric_cols[0]).head(10)
display(hardest)
hardest.plot(kind='bar', figsize=(8, 4), ylim=(0, 1), title='Hardest questions')
plt.tight_layout()

In [ ]:
sample_stats = samples.groupby('model').agg(
    samples=('correct', 'size'),
    correct=('correct', 'sum'),
    errors=('error', lambda s: s.notna().sum()),
    avg_total_tokens=('total_tokens', 'mean'),
)
sample_stats['accuracy'] = sample_stats['correct'] / sample_stats['samples']
sample_stats.sort_values('accuracy', ascending=False)